In [0]:
# Questo notebook inizializza la tabella Silver meteo del progetto.
#
# Legge i dati storici già caricati dal Bootstrapper nella tabella:
# progetto_meteo.bronze.meteo_storico
#
# Poi li copia nella tabella Silver:
# progetto_meteo.silver.dati_aggiornati
#
# Questo passaggio serve solo nella fase iniziale del progetto.
# Dopo questa inizializzazione, la tabella silver.dati_aggiornati viene aggiornata
# ogni giorno dal notebook Patcher, che consolida i dati live raccolti da LiveForecast.
#
# La logica non trasforma i dati: seleziona solo le colonne finali
# e le salva nella Silver nello stesso formato previsto dal progetto.


# Imposto catalogo e schemi del progetto.
catalogo = "progetto_meteo"
schema_bronze = "bronze"
schema_silver = "silver"


# Imposto i nomi completi delle tabelle.
tabella_storico = f"{catalogo}.{schema_bronze}.meteo_storico"
tabella_dati_aggiornati = f"{catalogo}.silver.dati_aggiornati"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo i dati storici già caricati dal Bootstrapper.
# Questa tabella contiene lo storico Open-Meteo dal 01/01/2024
# più l'eventuale recupero della giornata di avvio fino all'ultima ora completa.
df_storico = spark.table(tabella_storico)


# Seleziono e riordino le colonne previste dalla tabella Silver.
# In questo modo dati_aggiornati mantiene una struttura stabile e coerente
# con il resto della pipeline.
df_dati_aggiornati = df_storico.select(
    "Citta",
    "Regione",
    "Area",
    "Data",
    "Ora",
    "Temp_Max",
    "Temp_Min",
    "Temp_Oraria",
    "Umidita",
    "Velocita_Vento",
    "Precipitazioni",
    "Timestamp"
)


# Sovrascrivo la tabella Silver con lo storico iniziale.
# Uso overwrite perché questo notebook serve solo a inizializzare dati_aggiornati
# dopo il Bootstrapper.
df_dati_aggiornati.write.mode("overwrite").format("delta").saveAsTable(tabella_dati_aggiornati)


# Controllo finale.
# Mostro un campione della Silver appena inizializzata.
display(
    spark.table(tabella_dati_aggiornati)
    .limit(100)
)


# Stampo un riepilogo finale della clonazione.
print(f"Clonazione completata: {tabella_storico} → {tabella_dati_aggiornati}")
print(f"Righe copiate in dati_aggiornati: {spark.table(tabella_dati_aggiornati).count()}")